In [73]:
!pip install -q langchain langchain-core langchain-community \
langchain-text-splitters langchain-chroma langchain-groq \
chromadb sentence-transformers scikit-learn pandas numpy requests pydantic

In [93]:
import os
import re
import uuid
import json
import requests
import pandas as pd
from collections import Counter
from typing import Optional
from google.colab import userdata
os.environ["GROQ_API_KEY"]     = userdata.get("API_KEY")
os.environ["NGROK_AUTH_TOKEN"] = userdata.get("AUTH_TOKEN")
from pydantic import BaseModel, Field
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
from langchain_core.documents import Document
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from datetime import datetime


print("All imports successful.")

All imports successful.


In [ ]:
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=os.environ["GROQ_API_KEY"]
)

response = llm.invoke("say hello")

hf_embedder = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
st_embedder = SentenceTransformer("all-MiniLM-L6-v2")
print(response.content)
print("LLM and embedders ready.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Hello! It's nice to talk to you. Is there something I can help you with or would you like to chat?
LLM and embedders ready.


In [90]:
def load_any_source(source: str):
    temp_file = None
    docs = []

    if isinstance(source, list):
        for s in source:
            docs.extend(load_any_source(s))
        return docs

    if not isinstance(source, str):
        print(f"Invalid source type: {type(source)}")
        return []

    try:
        if source.startswith("http"):
            headers = {"User-Agent": "Mozilla/5.0"}
            response = requests.get(source, headers=headers)

            if response.status_code != 200:
                raise ValueError(f"Failed to fetch URL: {source}")

            content_type = response.headers.get("Content-Type", "")

            if "pdf" in content_type or source.endswith(".pdf"):
                if not response.content.startswith(b"%PDF"):
                    raise ValueError("Not a valid PDF file")

                temp_file = f"temp_{uuid.uuid4().hex}.pdf"
                with open(temp_file, "wb") as f:
                    f.write(response.content)

                loader = PyPDFLoader(temp_file)
                docs = loader.load()

            elif "csv" in content_type or source.endswith(".csv"):
                temp_file = f"temp_{uuid.uuid4().hex}.csv"
                with open(temp_file, "wb") as f:
                    f.write(response.content)

                df = pd.read_csv(temp_file)

                for _, row in df.iterrows():
                    docs.append(Document(page_content=str(row.to_dict())))

            elif "json" in content_type or source.endswith(".json"):
                try:
                    data = response.json()
                except:
                    docs.append(Document(page_content=response.text))
                    return docs

                if isinstance(data, list):
                    for item in data:
                        docs.append(Document(page_content=str(item)))
                else:
                    docs.append(Document(page_content=str(data)))

            else:
                docs.append(Document(page_content=response.text))

        else:
            if not os.path.exists(source):
                raise ValueError(f"Invalid file path: {source}")

            if source.endswith(".pdf"):
                loader = PyPDFLoader(source)
                docs = loader.load()

            elif source.endswith(".csv"):
                df = pd.read_csv(source)

                for _, row in df.iterrows():
                    docs.append(Document(page_content=str(row.to_dict())))

            elif source.endswith(".json"):
                with open(source) as f:
                    data = json.load(f)

                if isinstance(data, list):
                    for item in data:
                        docs.append(Document(page_content=str(item)))
                else:
                    docs.append(Document(page_content=str(data)))

            elif source.endswith(".txt"):
                with open(source, encoding="utf-8", errors="ignore") as f:
                    docs.append(Document(page_content=f.read()))

            else:
                raise ValueError(f"Unsupported file type: {source}")

        return docs

    except Exception as e:
        print(f"Error loading {source}: {e}")
        return []

    finally:
        if temp_file and os.path.exists(temp_file):
            os.remove(temp_file)

In [91]:
def chunk_documents(pages: list, chunk_size: int = 800, overlap: int = 100):
    if not pages:
        print("Warning: No pages provided to chunker.")
        return []
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=overlap,
        separators=["\n\n", "\n", " ", ""]
    )
    chunks = splitter.split_documents(pages)
    print(f"{len(chunks)} chunks created from {len(pages)} pages.")
    return chunks

In [ ]:
def build_vectorstore(chunks: list, persist_dir: str = "./chroma_db") -> Optional[Chroma]:
    if not chunks:
        print("Error: Cannot build vector store with 0 chunks. Check if PDF text was extracted correctly.")
        return None
    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=hf_embedder,
        persist_directory=persist_dir
    )
    print(f"Vector store built with {vectorstore._collection.count()} vectors.")
    return vectorstore

def retrieve_relevant_chunks(vectorstore: Chroma, query: str, k: int = 12) -> list:
    if vectorstore is None:
        return []
    results = vectorstore.similarity_search_with_score(query, k=k)
    filtered = [(doc, score) for doc, score in results if score < 1.2]
    return filtered

In [92]:
class GroundedData(BaseModel):
    summary:    str        = Field(..., description="2-3 line factual summary from the text only")
    key_points: list[str]  = Field(..., description="3-5 grounded bullet points from the text")
    entities:   list[str]  = Field(..., description="Named entities: people, models, methods, datasets")
    domain:     str        = Field(..., description="Academic domain e.g. NLP, cognitive psychology")
    claims:     list[str]  = Field(default=[], description="Verifiable factual claims made in the text")
    methodology: Optional[str] = Field(None, description="Research method or approach described, if any")

EXTRACTION_PROMPT = ChatPromptTemplate.from_template("""
You are a precision knowledge extractor for domain-specific LLM training data, integrated with the Alactic AGI automated grounding engine.

Source Document: {doc_name}
Related Context (from other documents in the corpus): {cross_doc_context}

Text to Extract From:
{text}

STRICT RULES:
1. Only use information present in the text above
2. Do NOT invent, infer, or hallucinate any facts
3. Cross-document context is for reference only — do NOT extract from it
4. Be factual, concise, and grounded

Extract the following:
- summary: 2–3 sentence factual summary
- key_points: 3–5 bullet points of core ideas
- entities: all named entities (people, models, techniques, datasets, institutions)
- domain: the academic/technical domain (e.g., NLP, cognitive psychology)
- claims: verifiable factual claims (e.g., \"BERT achieves 86.7 on GLUE\")
- methodology: the research method or experimental setup described (if any)

Return structured output only.
""")

extraction_chain = EXTRACTION_PROMPT | llm.with_structured_output(GroundedData)

In [ ]:
def compute_rag_confidence(
    result: GroundedData,
    vectorstore: Chroma,
    source_text: str
) -> float:
    score = 0.0

    if result.summary and len(result.summary) > 30:   score += 0.15
    if len(result.key_points) >= 3:                   score += 0.10
    if result.claims:
        grounded = 0
        for claim in result.claims[:3]:
            hits = vectorstore.similarity_search_with_score(claim, k=1)
            if hits and hits[0][1] < 0.9:
                grounded += 1
        score += 0.35 * (grounded / min(len(result.claims), 3))

    if len(result.entities) >= 5:   score += 0.20
    elif len(result.entities) >= 3: score += 0.10

    if len(result.summary) > 100:   score += 0.20
    elif len(result.summary) > 50:  score += 0.10

    return round(min(score, 1.0), 2)

In [ ]:
def deduplicate(dataset: list, threshold: float = 0.85) -> list:
    unique, embeddings = [], []
    for item in dataset:
        emb = st_embedder.encode(item["text"])
        if not any(
            cosine_similarity([emb], [e])[0][0] > threshold
            for e in embeddings
        ):
            unique.append(item)
            embeddings.append(emb)
    removed = len(dataset) - len(unique)
    print(f"Deduplication: {len(dataset)} → {len(unique)} ({removed} removed)")
    return unique

In [ ]:
DOMAIN_QUERIES = [
    "attention mechanism transformer self-attention",
    "cognitive load working memory psychology",
    "neural network training optimization",
    "human decision making bias heuristics",
    "language model pretraining fine-tuning",
    "experimental methodology research design",
]

def get_cross_doc_context(vectorstore: Chroma, chunk_text: str, k: int = 3) -> str:
    """
    Retrieve semantically similar chunks from OTHER documents
    to give the LLM cross-document grounding context.
    """
    results = vectorstore.similarity_search_with_score(chunk_text, k=k + 1)
    cross_doc = [
        doc.page_content[:300]
        for doc, score in results[1:]
        if score < 1.0
    ]
    return " | ".join(cross_doc) if cross_doc else "No cross-document context available."

In [95]:
def alactic_agi_layer(query: str, vectorstore: Chroma) -> dict:
    """
    Alactic AGI Integration Layer:
    Automated grounding of unstructured research data for domain-specific LLMs.
    """
    relevant = retrieve_relevant_chunks(vectorstore, query, k=8)
    if not relevant:
        return {"error": "No grounded context found."}
    context_blocks = []
    citations = []
    for i, (doc, score) in enumerate(relevant):
        context_blocks.append(f"[{i+1}] {doc.page_content[:400]}")
        citations.append({
            "chunk": i + 1,
            "source": doc.metadata.get("source", "unknown"),
            "page":   doc.metadata.get("page", "N/A"),
            "retrieval_score": round(float(score), 4)
        })

    grounded_context = "\n\n".join(context_blocks)
    cross_doc_ctx    = get_cross_doc_context(vectorstore, query)

    alactic_prompt = ChatPromptTemplate.from_template("""
    You are an Alactic AGI domain-specific reasoning engine.
    Your role: automated grounding of unstructured research data.

    STRICT RULES:
    - Only answer using the grounded context below
    - Do NOT hallucinate or infer beyond the provided text
    - Cite chunk numbers [1], [2] etc. in your answer
    - If context is insufficient, say so explicitly

    Query: {query}

    Grounded Context:
    {context}

    Cross-Document Intelligence:
    {cross_doc}

    Provide:
    1. A grounded, cited answer
    2. The domain this belongs to
    3. Confidence level (high / medium / low)
    """)

    chain    = alactic_prompt | llm
    response = chain.invoke({
        "query":     query,
        "context":   grounded_context,
        "cross_doc": cross_doc_ctx
    })

    return {
        "query":            query,
        "answer":           response.content,
        "citations":        citations,
        "cross_doc_context": cross_doc_ctx,
        "powered_by":       "Alactic AGI Reasoning Layer",
        "grounding_status": "VERIFIED",
        "timestamp":        datetime.now().isoformat()
    }

In [77]:
def run_grounded_rag_pipeline(
    sources: list[str] = None,
    domain_queries: list[str] = DOMAIN_QUERIES,
    confidence_threshold: float = 0.5,
    dedup_threshold: float = 0.85,
    output_path: str = None
) -> tuple:

    # ✅ Take input if not provided (NO extra logic)
    if not sources:
        raw = input("Enter PDF paths/URLs (comma-separated): ").strip()
        sources = [s.strip() for s in raw.split(",") if s.strip()]

    # ✅ Dynamic output path
    if output_path is None:
        os.makedirs("outputs", exist_ok=True)
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        output_path = f"outputs/grounded_data_{timestamp}.csv"

    print("\n" + "="*60)
    print("  GROUNDED RAG KNOWLEDGE ENGINE")
    print("="*60)

    print("\n[1/6] Loading documents...")
    pages = load_any_source(sources)

    if not pages:
        return pd.DataFrame(), None, "Pipeline failed: No documents loaded."

    print("\n[2/6] Chunking documents...")
    chunks = chunk_documents(pages)

    print("\n[3/6] Building RAG vector store...")
    vectorstore = build_vectorstore(chunks)

    if vectorstore is None:
        return pd.DataFrame(), None, "Pipeline failed: No text chunks extracted."

    print("\n[4/6] Running RAG-targeted extraction...")
    dataset = []
    seen_chunk_ids = set()

    for query in domain_queries:
        print(f" Query: '{query}'")
        relevant = retrieve_relevant_chunks(vectorstore, query, k=12)

        for doc, retrieval_score in relevant:
            chunk_id = hash(doc.page_content[:100])

            if chunk_id in seen_chunk_ids:
                continue
            seen_chunk_ids.add(chunk_id)

            clean_text = re.sub(r'\s+', ' ', doc.page_content).strip()

            if len(clean_text) < 100:
                continue

            cross_ctx = get_cross_doc_context(vectorstore, clean_text)

            try:
                result = extraction_chain.invoke({
                    "text": clean_text,
                    "doc_name": doc.metadata.get("doc_name", "unknown"),
                    "cross_doc_context": cross_ctx
                })

                if not (result and result.summary and len(result.key_points) >= 2):
                    continue

                confidence = compute_rag_confidence(result, vectorstore, clean_text)

                if confidence >= confidence_threshold:
                    dataset.append({
                        "text": clean_text,
                        "summary": result.summary,
                        "key_points": " | ".join(result.key_points),
                        "entities": ", ".join(result.entities),
                        "domain": result.domain,
                        "claims": " | ".join(result.claims),
                        "methodology": result.methodology or "",
                        "confidence": confidence,
                        "retrieval_score": round(float(retrieval_score), 4),
                        "query_trigger": query,
                        "page": doc.metadata.get("page"),
                        "source": doc.metadata.get("source"),
                        "doc_name": doc.metadata.get("doc_name"),
                    })

            except Exception as e:
                print(f"Extraction error: {e}")

    print(f"\n[5/6] Deduplicating {len(dataset)} records...")
    dataset = deduplicate(dataset, threshold=dedup_threshold)

    df = pd.DataFrame(dataset)

    if df.empty:
        print("No data extracted after processing and deduplication.")
        return df, vectorstore, ""

    print("\n[6/6] Generating final corpus summary...")

    df = df.sort_values("confidence", ascending=False)

    all_summaries = " ".join(df["summary"].tolist()[:30])

    synthesis_prompt = ChatPromptTemplate.from_template("""
    Synthesize these document summaries into one coherent 4-5 sentence overview.
    Summaries: {text}
    """)

    synthesis_chain = synthesis_prompt | llm
    final_summary = synthesis_chain.invoke({"text": all_summaries})

    df.to_csv(output_path, index=False)

    print("\nPIPELINE COMPLETE")
    print(f"Saved to: {output_path}")

    return df, vectorstore, final_summary.content

In [96]:
df, vectorstore, corpus_summary = run_grounded_rag_pipeline()


Enter PDF paths/URLs (comma-separated): https://d8it4huxumps7.cloudfront.net/uploads/attachements/files/c88b059e-f871-48a6-87ad-b5709220dc4d.pdf

  GROUNDED RAG KNOWLEDGE ENGINE

[1/6] Loading documents...

[2/6] Chunking documents...
22 chunks created from 7 pages.

[3/6] Building RAG vector store...
Vector store built with 1008 vectors.

[4/6] Running RAG-targeted extraction...
 Query: 'attention mechanism transformer self-attention'
 Query: 'cognitive load working memory psychology'
 Query: 'neural network training optimization'
 Query: 'human decision making bias heuristics'
 Query: 'language model pretraining fine-tuning'
 Query: 'experimental methodology research design'

[5/6] Deduplicating 4 records...
Deduplication: 4 → 4 (0 removed)

[6/6] Generating final corpus summary...

PIPELINE COMPLETE
Saved to: outputs/grounded_data_20260328_225507.csv
